# 08 — Swappable backends

*The architectural payoff of [ADR-0009](../docs/arc42/09-decisions/0009-adopt-ddd-ubiquitous-language-and-hexagonal-lite.md) + [ADR-0010](../docs/arc42/09-decisions/0010-refine-positioning-to-educational-first-production-capable.md) + [ADR-0011](../docs/arc42/09-decisions/0011-kernel-backend-port-api.md): the same Domain code runs on the reference CPU kernel or on Eigen's SIMD-and-GEMM kernels by changing one CMake variable.*

By the end of this notebook:

1. You can read and explain the `KernelBackend` port surface in `tensor::core::concepts`.
2. You understand why per-op named methods (not generic dispatch) was the chosen design — read [ADR-0011 §Pros and Cons](../docs/arc42/09-decisions/0011-kernel-backend-port-api.md).
3. You can see the same `dot(W, x)` produce identical output on `reference::Backend` and `eigen::Backend`.
4. You see *why* the architecture refines ADR-0001 from "educational-only" to "educational-first, production-capable".

In [ ]:
#pragma cling add_include_path("../include")
#include <iostream>
#include <iomanip>
#include <tensor/core/core.hpp>
#include <tensor/core/backend/reference.hpp>

// Eigen adapter is conditional on the CMake variable. When the notebook
// runs against an Eigen-enabled build, this include resolves; otherwise
// the test cells below report "Eigen not enabled in this build".
#if __has_include(<tensor/core/backend/eigen.hpp>) && defined(TENSOR_HAS_EIGEN)
#  include <tensor/core/backend/eigen.hpp>
#  define HAVE_EIGEN 1
#else
#  define HAVE_EIGEN 0
#endif

using tensor::core::Axis;
using tensor::core::DynamicShape;
using tensor::core::DynamicTensor;
using RefBackend = tensor::core::backend::reference::Backend;

## 1 — The port

Read `include/tensor/core/concepts.hpp`. `KernelBackend` is a C++20 concept that names 15 methods every adapter must provide:

- Element-wise binary: `add` / `sub` / `mul` / `div`
- Element-wise unary: `exp` / `log` / `relu` / `neg`
- Broadcast element-wise: `broadcast_add` / `broadcast_sub` / `broadcast_mul`
- Contraction: `contract` (consumes a `ContractPlan` computed by the Domain)
- Reduction: `reduce_sum`
- Reverse-mode helper: `unbroadcast`

Every method returns a fresh `DynamicTensor<T>`. The Domain computes `BroadcastPlan` and `ContractPlan` (which describe *which axes are shared, which collapse, which survive*) and hands them to the backend. The backend executes the lowered, planned operation. Named-axis algebra stays in the Domain; math kernels stay in the adapter.

## 2 — The reference adapter

`tensor::core::backend::reference::Backend` is the always-available default. Its implementation is a thin wrapper around a row-major flat-vector loop per method. Below: build a small matrix-vector multiply via `contract` and observe the result.

```
y_i = Σ_j W_{ij} x_j
```

In [ ]:
RefBackend ref;
DynamicTensor<double> W(DynamicShape{Axis{"i", 2}, Axis{"j", 3}}, {1, 2, 3, 4, 5, 6});
DynamicTensor<double> x(DynamicShape{Axis{"j", 3}}, {10, 20, 30});
auto cplan = tensor::core::contract_plan(W.shape(), x.shape());
auto y_ref = ref.contract(W, x, cplan);
std::cout << "reference y = (" << y_ref[0] << ", " << y_ref[1] << ")" << std::endl;
std::cout << "expect (140, 320)" << std::endl;

## 3 — The Eigen adapter (when built with `TENSOR_KERNEL_BACKEND=eigen`)

`tensor::core::backend::eigen::Backend` fast-paths element-wise on `Eigen::Map<Eigen::ArrayXd>` (SIMD vectorised) and matvec / matmul on `Eigen::Matrix` (BLAS-flavoured GEMM when Eigen was built with BLAS). It delegates everything else (broadcast, higher-rank contractions, non-double element types) to the reference adapter so the port surface stays complete.

Build with:

```bash
cmake --preset=default -DTENSOR_KERNEL_BACKEND=eigen
```

Output below should match `reference` byte-for-byte (or within 1e-9).

In [ ]:
#if HAVE_EIGEN
tensor::core::backend::eigen::Backend eig;
auto y_eig = eig.contract(W, x, cplan);
std::cout << "eigen y = (" << y_eig[0] << ", " << y_eig[1] << ")" << std::endl;
std::cout << "match: " << (y_eig[0] == y_ref[0] && y_eig[1] == y_ref[1]) << std::endl;
#else
std::cout << "Eigen adapter not enabled in this build.\n"
          << "Configure with cmake --preset=default -DTENSOR_KERNEL_BACKEND=eigen and rebuild." << std::endl;
#endif

## 4 — Why per-op named methods (not generic dispatch or visitor)

ADR-0011 §Pros and Cons walks through three designs:

1. **Generic dispatch** (`apply<Op>(a, b)`) — tiny port surface; each adapter writes one method. Forces every backend through one dispatch path; defeats Eigen's expression templates; cannot specialise matmul separately from add.
2. **Per-op methods** (chosen) — each backend implements only the ops it cares about; Eigen, BLAS, WebGPU each get their natural fast path; ~15 methods of boilerplate, mitigated by the upcoming `DefaultBackend<Derived>` CRTP helper that provides reference implementations for unspecialised methods.
3. **Visitor over Expression** — maximally flexible (fusion, JIT, scheduling); massive design + implementation cost; mismatched with the educational identity.

The per-op design is why `eigen::Backend::add` can fall through to `Eigen::ArrayXd + Eigen::ArrayXd` (one line) instead of routing through a generic dispatcher that erases the op.

## 5 — Educational-first, production-capable

ADR-0001 originally framed the library as "educational, not production". The Phase 2 build-out showed that two of its three rationale points (speed; the third was coverage and the fourth was support / ABI commitments) collapse cleanly when the architecture admits production-grade backend adapters. ADR-0010 refines (not supersedes) ADR-0001:

- The Domain (named-axis algebra, autograd, TeX bridge) remains educational — small, readable, unencumbered.
- Production users can adopt the Eigen / WebGPU / Kokkos adapter when ready, **as-is**: no ABI commitment, no coverage parity, no formal support.
- Identity intact; production becomes a permitted side effect, not the headline.

This notebook is the architectural payoff. The next adapter is WebGPU (Phase 3); the same pattern applies.

## Where to go from here

- **Phase 3 — WebGPU adapter** ([ADR-0006](../docs/arc42/09-decisions/0006-adopt-webgpu-as-gpu-backend.md)). Same per-op design; `Backend` lives at `include/tensor/core/backend/webgpu.hpp` once we ship it. WGSL kernel emission from named-axis expressions.
- **Phase 1.5 mop-up** — `_ax` NTTP compile-time path, `_tex` consteval bridge, mdspan polyfill (`Kokkos::` namespace adapter), xeus-cling notebook CI, `Variable::zero_grad()` ergonomics.
- **Phase 4** — `0.1.0` public release: full tutorial corpus, Jupyter Book site on GitHub Pages, LyX export module.